<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_RegressionLineageHarness_v13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FHIRy–pyOMOP TFL Regression and Lineage Harness

This notebook implements submission-blocking regression tests for the versioned Transformation Fidelity Layer.

It tests:

- deterministic warning responses,
- absence of unrelated warning contamination,
- transformation ID uniqueness,
- source-key recoverability,
- target-link completeness,
- target-to-source consistency, and
- explicit traceability-loss consistency.



# Phase A

## Load frozen schema and corrected audits

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")
RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
FREEZE_ROOT = RUN_ROOT / "submission_freeze_v12"
SCHEMA_DIR = FREEZE_ROOT / "schema"
MANIFEST_DIR = FREEZE_ROOT / "manifests"

V10_AUDIT_DIR = (
    RUN_ROOT
    / "encounter_audit_repair_v10"
    / "fidelity_audit_corrected"
)

OUTPUT_DIR = FREEZE_ROOT / "regression_v13"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = ["V0", "V1", "V2", "V3", "V4", "V5"]

schema_path = SCHEMA_DIR / "tfl_audit_schema_v1.0.0.json"
warning_path = SCHEMA_DIR / "tfl_warning_vocabulary_v1.0.0.csv"
status_path = SCHEMA_DIR / "tfl_fidelity_status_v1.0.0.csv"

for path in [schema_path, warning_path, status_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. Run v12 first."
        )

TFL_SCHEMA = json.loads(schema_path.read_text(encoding="utf-8"))
WARNING_VOCAB = pd.read_csv(warning_path)
STATUS_VOCAB = pd.read_csv(status_path)

WARNING_CODES = set(WARNING_VOCAB["warning_code"].astype(str))
FIDELITY_STATUS_VALUES = set(
    STATUS_VOCAB["fidelity_status"].astype(str)
)

AUDITS = {}

for variant in VARIANTS:
    path = V10_AUDIT_DIR / f"{variant}_fidelity_audit_v10.parquet"

    if not path.exists():
        raise FileNotFoundError(path)

    AUDITS[variant] = pd.read_parquet(path)

print("Loaded corrected audits:", ", ".join(VARIANTS))

Mounted at /content/drive
Loaded corrected audits: V0, V1, V2, V3, V4, V5


# Phase B

## Warning regression tests

In [2]:
def split_warnings(value):
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "null"}:
        return []

    return [item for item in text.split("|") if item]


warning_rows = []

for variant, audit in AUDITS.items():
    counter = defaultdict(int)

    for value in audit["warning_code"]:
        for warning in split_warnings(value):
            counter[warning] += 1

    for warning in sorted(WARNING_CODES):
        warning_rows.append({
            "variant": variant,
            "warning_code": warning,
            "warning_count": int(counter.get(warning, 0)),
        })

WARNING_COUNTS = pd.DataFrame(warning_rows)

warning_matrix = WARNING_COUNTS.pivot(
    index="variant",
    columns="warning_code",
    values="warning_count",
).fillna(0).astype(int)

display(warning_matrix)

warning_code,W_CONFLICTING_CODING,W_DEMOGRAPHIC_MISSING,W_DUPLICATE_SOURCE_ID,W_MEDICATION_ATTRIBUTION,W_TRACEABILITY_LOSS,W_UNMAPPED_RESOURCE,W_UNRESOLVED_REFERENCE
variant,,,,,,,
V0,0,0,0,9687,0,28262,0
V1,0,161,0,9687,0,28262,0
V2,0,0,2393,9687,2393,28262,0
V3,2510,0,0,9687,0,28262,0
V4,0,0,0,9843,0,25436,0
V5,2510,161,0,9687,0,28262,0


In [3]:
# Exact deterministic counts from the finalized v10 controlled run.
# These values are intentionally frozen as regression expectations.
EXPECTED_COUNTS = {
    "W_DEMOGRAPHIC_MISSING": {
        "V0": 0,
        "V1": 161,
        "V2": 0,
        "V3": 0,
        "V4": 0,
        "V5": 161,
    },
    "W_DUPLICATE_SOURCE_ID": {
        "V0": 0,
        "V1": 0,
        "V2": 2393,
        "V3": 0,
        "V4": 0,
        "V5": 0,
    },
    "W_TRACEABILITY_LOSS": {
        "V0": 0,
        "V1": 0,
        "V2": 2393,
        "V3": 0,
        "V4": 0,
        "V5": 0,
    },
    "W_CONFLICTING_CODING": {
        "V0": 0,
        "V1": 0,
        "V2": 0,
        "V3": 2510,
        "V4": 0,
        "V5": 2510,
    },
    "W_MEDICATION_ATTRIBUTION": {
        "V0": 9687,
        "V1": 9687,
        "V2": 9687,
        "V3": 9687,
        "V4": 9843,
        "V5": 9687,
    },
}

regression_rows = []

for warning, expected_by_variant in EXPECTED_COUNTS.items():
    if warning not in warning_matrix.columns:
        raise RuntimeError(
            f"Expected warning is absent from audit outputs: {warning}"
        )

    for variant, expected in expected_by_variant.items():
        actual = int(
            warning_matrix.loc[
                variant,
                warning,
            ]
        )

        regression_rows.append({
            "variant": variant,
            "warning_code": warning,
            "expected_count": expected,
            "actual_count": actual,
            "pass": actual == expected,
        })

WARNING_REGRESSION = pd.DataFrame(regression_rows)

display(
    WARNING_REGRESSION[
        ~WARNING_REGRESSION["pass"]
    ]
)

if not WARNING_REGRESSION["pass"].all():
    raise RuntimeError(
        "One or more deterministic warning regression tests failed."
    )

print("PASS: all frozen warning counts reproduce exactly.")

,variant,warning_code,expected_count,actual_count,pass


PASS: all frozen warning counts reproduce exactly.


## Unrelated-warning contamination

In [4]:
# For the controlled warning families, unrelated single perturbations
# must remain at their V0 count. V5 is intentionally combined V1+V3.
CONTROLLED_WARNING_FAMILIES = [
    "W_DEMOGRAPHIC_MISSING",
    "W_DUPLICATE_SOURCE_ID",
    "W_TRACEABILITY_LOSS",
    "W_CONFLICTING_CODING",
    "W_MEDICATION_ATTRIBUTION",
]

EXPECTED_ACTIVE = {
    "V1": {"W_DEMOGRAPHIC_MISSING"},
    "V2": {"W_DUPLICATE_SOURCE_ID", "W_TRACEABILITY_LOSS"},
    "V3": {"W_CONFLICTING_CODING"},
    "V4": {"W_MEDICATION_ATTRIBUTION"},
    "V5": {"W_DEMOGRAPHIC_MISSING", "W_CONFLICTING_CODING"},
}

contamination_rows = []

for variant in ["V1", "V2", "V3", "V4", "V5"]:
    for warning in CONTROLLED_WARNING_FAMILIES:
        if warning in EXPECTED_ACTIVE[variant]:
            continue

        baseline = int(warning_matrix.loc["V0", warning])
        actual = int(warning_matrix.loc[variant, warning])

        contamination_rows.append({
            "variant": variant,
            "warning_code": warning,
            "v0_count": baseline,
            "variant_count": actual,
            "delta": actual - baseline,
            "tolerance": 0,
            "pass": actual == baseline,
        })

CONTAMINATION_CHECK = pd.DataFrame(contamination_rows)

display(
    CONTAMINATION_CHECK[
        ~CONTAMINATION_CHECK["pass"]
    ]
)

if not CONTAMINATION_CHECK["pass"].all():
    raise RuntimeError(
        "An unrelated controlled warning family changed unexpectedly."
    )

print("PASS: no cross-warning contamination in V1-V5.")

,variant,warning_code,v0_count,variant_count,delta,tolerance,pass


PASS: no cross-warning contamination in V1-V5.


# Phase C

## Lineage integrity tests

In [5]:
SOURCE_KEY = [
    "source_resource_type",
    "source_df_index",
]

lineage_rows = []

for variant, audit in AUDITS.items():
    work = audit.copy()

    # Required source key for record-level audit.
    source_key_complete = (
        work["source_resource_type"].notna()
        & work["source_df_index"].notna()
    )

    # transformation_id must be present and unique.
    transformation_present = (
        work["transformation_id"].notna()
        & (
            work["transformation_id"]
            .astype(str)
            .str.len()
            > 0
        )
    )

    duplicate_transformation_ids = int(
        work.loc[
            transformation_present,
            "transformation_id",
        ]
        .astype(str)
        .duplicated(keep=False)
        .sum()
    )

    if duplicate_transformation_ids:
        raise RuntimeError(
            f"{variant}: duplicated transformation_id rows = "
            f"{duplicate_transformation_ids:,}"
        )

    if not source_key_complete.all():
        raise RuntimeError(
            f"{variant}: {int((~source_key_complete).sum()):,} rows "
            "have an incomplete source key."
        )

    mapping_event = (
        work["is_mapping_event"]
        .fillna(False)
        .astype(bool)
    )

    traceability_loss = (
        work["fidelity_status"]
        .astype(str)
        .eq("TRACEABILITY_LOSS")
    )

    target_complete = (
        work["target_omop_table"].notna()
        & work["target_omop_record_id"].notna()
    )

    nonloss_mapping = (
        mapping_event
        & ~traceability_loss
    )

    missing_target_nonloss = int(
        (
            nonloss_mapping
            & ~target_complete
        ).sum()
    )

    if missing_target_nonloss:
        raise RuntimeError(
            f"{variant}: {missing_target_nonloss:,} non-loss mapping events "
            "do not have a complete target link."
        )

    loss_warning_present = work["warning_code"].apply(
        lambda value: "W_TRACEABILITY_LOSS" in split_warnings(value)
    )

    inconsistent_loss = int(
        (
            traceability_loss
            & ~loss_warning_present
        ).sum()
    )

    if inconsistent_loss:
        raise RuntimeError(
            f"{variant}: {inconsistent_loss:,} TRACEABILITY_LOSS rows "
            "do not carry W_TRACEABILITY_LOSS."
        )

    lineage_rows.append({
        "variant": variant,
        "audit_rows": len(work),
        "source_keys_complete": True,
        "transformation_ids_unique": True,
        "nonloss_mapping_targets_complete": True,
        "traceability_loss_warning_consistent": True,
        "traceability_loss_rows": int(traceability_loss.sum()),
    })

LINEAGE_INTEGRITY = pd.DataFrame(lineage_rows)
display(LINEAGE_INTEGRITY)

print("PASS: record-level lineage integrity checks passed.")

,variant,audit_rows,source_keys_complete,transformation_ids_unique,nonloss_mapping_targets_complete,traceability_loss_warning_consistent,traceability_loss_rows
0,V0,179333,True,True,True,True,0
1,V1,179333,True,True,True,True,0
2,V2,179333,True,True,True,True,2393
3,V3,179333,True,True,True,True,0
4,V4,176507,True,True,True,True,0
5,V5,179333,True,True,True,True,0


PASS: record-level lineage integrity checks passed.


## Target-to-source consistency

In [6]:
target_consistency_rows = []

for variant, audit in AUDITS.items():
    work = audit.copy()

    work["source_df_index"] = pd.to_numeric(
        work["source_df_index"],
        errors="raise",
    ).astype("int64")

    mapping = work[
        work["is_mapping_event"]
        .fillna(False)
        .astype(bool)
        & work["target_omop_table"].notna()
        & work["target_omop_record_id"].notna()
    ].copy()

    mapping["source_key"] = (
        mapping["source_resource_type"].astype(str)
        + "|"
        + mapping["source_df_index"].astype(str)
    )

    # One target record should not silently resolve to multiple source rows.
    target_group = (
        mapping.groupby(
            [
                "target_omop_table",
                "target_omop_record_id",
            ],
            dropna=False,
        )["source_key"]
        .nunique()
        .rename("unique_source_keys")
        .reset_index()
    )

    conflicting_targets = target_group[
        target_group["unique_source_keys"] > 1
    ]

    if len(conflicting_targets):
        display(conflicting_targets.head(20))
        raise RuntimeError(
            f"{variant}: {len(conflicting_targets):,} target OMOP records "
            "link to more than one source row."
        )

    target_consistency_rows.append({
        "variant": variant,
        "mapped_target_records": len(target_group),
        "targets_with_multiple_source_keys": 0,
        "pass": True,
    })

TARGET_SOURCE_CONSISTENCY = pd.DataFrame(
    target_consistency_rows
)

display(TARGET_SOURCE_CONSISTENCY)

print("PASS: each mapped target record resolves to one source row.")

,variant,mapped_target_records,targets_with_multiple_source_keys,pass
0,V0,151071,0,True
1,V1,151071,0,True
2,V2,151071,0,True
3,V3,151071,0,True
4,V4,151071,0,True
5,V5,151071,0,True


PASS: each mapped target record resolves to one source row.


# Phase D

## Export regression report

In [7]:
WARNING_COUNTS.to_csv(
    OUTPUT_DIR / "warning_counts_v13.csv",
    index=False,
)

WARNING_REGRESSION.to_csv(
    OUTPUT_DIR / "warning_regression_v13.csv",
    index=False,
)

CONTAMINATION_CHECK.to_csv(
    OUTPUT_DIR / "warning_contamination_v13.csv",
    index=False,
)

LINEAGE_INTEGRITY.to_csv(
    OUTPUT_DIR / "lineage_integrity_v13.csv",
    index=False,
)

TARGET_SOURCE_CONSISTENCY.to_csv(
    OUTPUT_DIR / "target_source_consistency_v13.csv",
    index=False,
)

FINAL_GATE = pd.DataFrame([
    {
        "test_family": "warning_regression",
        "pass": bool(WARNING_REGRESSION["pass"].all()),
    },
    {
        "test_family": "warning_contamination",
        "pass": bool(CONTAMINATION_CHECK["pass"].all()),
    },
    {
        "test_family": "lineage_integrity",
        "pass": True,
    },
    {
        "test_family": "target_source_consistency",
        "pass": bool(TARGET_SOURCE_CONSISTENCY["pass"].all()),
    },
])

display(FINAL_GATE)

FINAL_GATE.to_csv(
    OUTPUT_DIR / "tfl_regression_gate_v13.csv",
    index=False,
)

if not FINAL_GATE["pass"].all():
    raise RuntimeError("TFL regression gate failed.")

print("PASS: submission regression and lineage gate complete.")
print("Outputs:", OUTPUT_DIR)

,test_family,pass
0,warning_regression,True
1,warning_contamination,True
2,lineage_integrity,True
3,target_source_consistency,True


PASS: submission regression and lineage gate complete.
Outputs: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/submission_freeze_v12/regression_v13


# Exit criteria

The harness passes only when the finalized warning counts reproduce exactly, unrelated warning families stay at the V0 baseline, transformation IDs remain unique, source keys are complete, non-loss mappings retain target links, and target OMOP records do not silently link to multiple source rows.

The next blocking work is medication-attribution hardening and an unresolved-reference controlled test.